# Fileset Scale Example

Upload thousands of files from a single directory in minutes

In [ ]:
%pip install -U "lightningrod-ai[transfer]" python-dotenv -q

from IPython.display import clear_output
clear_output()


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /Users/kskotheim/Desktop/dev/forecasting/lightningrod-python-sdk/.venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Setup

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

In [3]:
NUM_FILES = 2000
CHARS_PER_FILE = 50_000
MAX_WORKERS = 100

## Generate temp files

In [4]:
import shutil
import tempfile, time
from pathlib import Path

tmp_dir = Path(tempfile.mkdtemp())
dummy_text = ("A" * 100 + "\n") * (CHARS_PER_FILE // 101)
# pad to exact length
dummy_text = dummy_text[:CHARS_PER_FILE]

t0 = time.time()
for i in range(NUM_FILES):
    p = tmp_dir / f"file_{i:06d}.txt"
    p.write_text(dummy_text)
    if (i + 1) % 5000 == 0:
        print(f"Generated {i + 1}/{NUM_FILES} files")

gen_time = time.time() - t0
print(f"\nGenerated {NUM_FILES} files in {gen_time:.1f}s ({tmp_dir})")


Generated 2000 files in 0.6s (/var/folders/n5/1nmlnrf90y7dthhl9pk5w8h80000gn/T/tmpj5tjnmq1)


## Create fileset and upload

Uses GCS Transfer Manager with downscoped credentials for efficient parallel uploads. The credentials are write-only and scoped to the fileset folder.

In [ ]:
fileset = lr.filesets.create(
    name="Scale Test - 2K Files",
    description="Scale test: 2000 files x 50000 chars",
)
print(f"Created FileSet: {fileset.id}")

Created FileSet: 562a5602-428e-4072-97dd-51f399e2309e


In [6]:
t1 = time.time()
try:
    result = lr.filesets.upload_directory(
        fileset.id,
        tmp_dir,
        pattern="*.txt",
        max_workers=MAX_WORKERS,
        show_progress=True,
    )
    success_count = result.succeeded
    error_count = result.failed
    errors = result.errors
finally:
    shutil.rmtree(tmp_dir, ignore_errors=True)
    print(f"Deleted temp directory: {tmp_dir}")

upload_time = time.time() - t1
print(f"\nUpload complete: {success_count} succeeded, {error_count} failed in {upload_time:.1f}s")
if errors:
    print(f"Sample errors: {errors[:3]}")

/Users/kskotheim/Desktop/dev/forecasting/lightningrod-python-sdk/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Uploading 2000 files...
  Progress: 2000/2000 (100%) - 30.0 files/s - ETA: 0s
Deleted temp directory: /var/folders/n5/1nmlnrf90y7dthhl9pk5w8h80000gn/T/tmpj5tjnmq1

Upload complete: 2000 succeeded, 0 failed in 69.3s


## Verify upload count

In [7]:
# Verify upload count based on successful uploads
print(f"Files uploaded: {success_count}")
assert success_count == NUM_FILES, f"Expected {NUM_FILES}, got {success_count}"
print("All files uploaded successfully!")

Files uploaded: 2000
All files uploaded successfully!
